# Notebook 01 — V1: Mapa de la paradoja

**Visualización:** Choropleth map en Flourish  
**Pregunta:** ¿En qué países de Europa la brecha de bienestar mental entre hombres y mujeres es mayor?  
**Output:** `data/processed/v1_mapa_paradoja.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../')
OUT  = ROOT / 'data' / 'processed'

# Nombres completos de países (para etiquetas en Flourish)
COUNTRY_NAMES = {
    'AT': 'Austria', 'BE': 'Belgium', 'BG': 'Bulgaria',
    'CH': 'Switzerland', 'CY': 'Cyprus', 'CZ': 'Czechia',
    'DE': 'Germany', 'DK': 'Denmark', 'EE': 'Estonia',
    'ES': 'Spain', 'FI': 'Finland', 'FR': 'France',
    'GB': 'United Kingdom', 'GR': 'Greece', 'HR': 'Croatia',
    'HU': 'Hungary', 'IE': 'Ireland', 'IL': 'Israel',
    'IS': 'Iceland', 'IT': 'Italy', 'LT': 'Lithuania',
    'LV': 'Latvia', 'ME': 'Montenegro', 'NL': 'Netherlands',
    'NO': 'Norway', 'PL': 'Poland', 'PT': 'Portugal',
    'RS': 'Serbia', 'SE': 'Sweden', 'SI': 'Slovenia',
    'SK': 'Slovakia', 'UA': 'Ukraine'
}

print('Setup OK')

Setup OK


## 1. Cargar ESS R11 limpio

In [2]:
ess11 = pd.read_csv(OUT / 'ESS11_clean.csv', low_memory=False)
print(f'ESS R11: {ess11.shape[0]:,} filas × {ess11.shape[1]} columnas')
print(f'Países: {sorted(ess11["cntry"].unique())}')

ESS R11: 50,116 filas × 48 columnas
Países: ['AT', 'BE', 'BG', 'CH', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IL', 'IS', 'IT', 'LT', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RS', 'SE', 'SI', 'SK', 'UA']


## 2. Calcular medias de bienestar por país y género

In [3]:
# Media de IDX_WELLBEING por país y género
wb_gndr = (
    ess11.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
    .agg(['mean', 'count'])
    .reset_index()
)

# Pivotar: una fila por país
# gndr=1 → Hombres, gndr=2 → Mujeres (orden numérico ascendente en el pivot)
wb_wide = wb_gndr.pivot(index='cntry', columns='gndr', values=['mean', 'count'])
wb_wide.columns = ['wb_hombres', 'wb_mujeres', 'n_hombres', 'n_mujeres']
wb_wide = wb_wide.reset_index()

# Verificar que la asignación es correcta (hombres deben tener mayor n en países con sesgo muestral conocido)
assert wb_wide.loc[wb_wide['cntry'] == 'ES', 'n_hombres'].values[0] == \
       ess11[(ess11['cntry'] == 'ES') & (ess11['gndr'] == 1)].shape[0], \
       "Error: columna hombres/mujeres invertida"

# Calcular brecha y total  (positivo = hombres > mujeres)
wb_wide['BRECHA_GNDR'] = wb_wide['wb_hombres'] - wb_wide['wb_mujeres']
wb_wide['n_total'] = wb_wide['n_hombres'] + wb_wide['n_mujeres']

# Añadir nombre completo
wb_wide['country_name'] = wb_wide['cntry'].map(COUNTRY_NAMES)

# Ordenar columnas
v1 = wb_wide[['cntry', 'country_name', 'wb_hombres', 'wb_mujeres',
              'BRECHA_GNDR', 'n_hombres', 'n_mujeres', 'n_total']].copy()

# Redondear
for col in ['wb_hombres', 'wb_mujeres', 'BRECHA_GNDR']:
    v1[col] = v1[col].round(3)

print(f'Filas (países): {len(v1)}')
print(v1.sort_values('BRECHA_GNDR', ascending=False).to_string(index=False))

Filas (países): 30
cntry   country_name  wb_hombres  wb_mujeres  BRECHA_GNDR  n_hombres  n_mujeres  n_total
   PT       Portugal       6.312       6.031        0.281      578.0      795.0   1373.0
   CY         Cyprus       6.490       6.213        0.277      308.0      375.0    683.0
   ES          Spain       6.562       6.357        0.205      875.0      969.0   1844.0
   SK       Slovakia       6.232       6.052        0.180      661.0      771.0   1432.0
   IT          Italy       6.337       6.163        0.174     1332.0     1510.0   2842.0
   BE        Belgium       6.629       6.459        0.170      809.0      785.0   1594.0
   FR         France       6.522       6.355        0.167      873.0      896.0   1769.0
   RS         Serbia       6.329       6.165        0.164      728.0      831.0   1559.0
   UA        Ukraine       5.811       5.680        0.130      944.0     1713.0   2657.0
   GR         Greece       6.384       6.258        0.126     1239.0     1517.0   2756.0
  

## 3. Estadísticos descriptivos de la brecha

In [4]:
print('Descriptivos BRECHA_GNDR:')
print(v1['BRECHA_GNDR'].describe().round(3))
print()
print(f'Países con brecha positiva (H > M): {(v1["BRECHA_GNDR"] > 0).sum()}')
print(f'Países con brecha negativa (M > H): {(v1["BRECHA_GNDR"] < 0).sum()}')
print()
mediana = v1['BRECHA_GNDR'].median()
print(f'Mediana brecha: {mediana:.3f}')
print()
print('España:')
print(v1[v1['cntry'] == 'ES'].to_string(index=False))

Descriptivos BRECHA_GNDR:
count    30.000
mean      0.097
std       0.082
min      -0.047
25%       0.041
50%       0.100
75%       0.156
max       0.281
Name: BRECHA_GNDR, dtype: float64

Países con brecha positiva (H > M): 27
Países con brecha negativa (M > H): 3

Mediana brecha: 0.100

España:
cntry country_name  wb_hombres  wb_mujeres  BRECHA_GNDR  n_hombres  n_mujeres  n_total
   ES        Spain       6.562       6.357        0.205      875.0      969.0   1844.0


## 4. Añadir columna de cuartil para escala de color en Flourish

In [5]:
# Cuartil de brecha (para categorizar el color del mapa)
v1['cuartil_brecha'] = pd.qcut(
    v1['BRECHA_GNDR'],
    q=4,
    labels=['Q1 (menor)', 'Q2', 'Q3', 'Q4 (mayor)'],
    duplicates='drop'
)

# Flag España
v1['es_espana'] = v1['cntry'] == 'ES'

print('Distribución por cuartil:')
print(v1.groupby('cuartil_brecha')['cntry'].apply(list))

Distribución por cuartil:
cuartil_brecha
Q1 (menor)    [DE, EE, FI, IE, IS, LT, LV, ME]
Q2                [AT, BG, CH, GB, HU, NO, SE]
Q3                [GR, HR, IL, NL, PL, SI, UA]
Q4 (mayor)    [BE, CY, ES, FR, IT, PT, RS, SK]
Name: cntry, dtype: object


## 5. Verificar y guardar

In [6]:
def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)}')
    print(f'Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos:\n{nulos_con}')
    else:
        print('Sin valores nulos')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    print(f'Muestra:\n{df.head(3).to_string()}')
    print('✓ OK')

verificar_csv_flourish(v1, 'v1_mapa_paradoja')


=== v1_mapa_paradoja ===
Filas: 30
Columnas: ['cntry', 'country_name', 'wb_hombres', 'wb_mujeres', 'BRECHA_GNDR', 'n_hombres', 'n_mujeres', 'n_total', 'cuartil_brecha', 'es_espana']
Sin valores nulos
Muestra:
  cntry country_name  wb_hombres  wb_mujeres  BRECHA_GNDR  n_hombres  n_mujeres  n_total cuartil_brecha  es_espana
0    AT      Austria       6.596       6.530        0.066      990.0     1360.0   2350.0             Q2      False
1    BE      Belgium       6.629       6.459        0.170      809.0      785.0   1594.0     Q4 (mayor)      False
2    BG     Bulgaria       6.261       6.188        0.073     1062.0     1172.0   2234.0             Q2      False
✓ OK


In [7]:
out_path = OUT / 'v1_mapa_paradoja.csv'
v1.to_csv(out_path, index=False)
print(f'Guardado: {out_path}')
print(f'Filas: {len(v1)}, Columnas: {list(v1.columns)}')

Guardado: ../data/processed/v1_mapa_paradoja.csv
Filas: 30, Columnas: ['cntry', 'country_name', 'wb_hombres', 'wb_mujeres', 'BRECHA_GNDR', 'n_hombres', 'n_mujeres', 'n_total', 'cuartil_brecha', 'es_espana']


## 6. Resumen para Flourish

In [8]:
print('=' * 60)
print('RESUMEN V1 — MAPA DE LA PARADOJA')
print('=' * 60)
print()
print(f'Países representados: {len(v1)}')
print(f'Rango brecha: [{v1["BRECHA_GNDR"].min():.3f}, {v1["BRECHA_GNDR"].max():.3f}]')
print(f'Mediana brecha: {v1["BRECHA_GNDR"].median():.3f}')
print()
print('Top 5 mayor brecha:')
print(v1.nlargest(5, 'BRECHA_GNDR')[['cntry','country_name','wb_hombres','wb_mujeres','BRECHA_GNDR']].to_string(index=False))
print()
print('Top 5 menor brecha (o invertida):')
print(v1.nsmallest(5, 'BRECHA_GNDR')[['cntry','country_name','wb_hombres','wb_mujeres','BRECHA_GNDR']].to_string(index=False))
print()
print('Configuración Flourish:')
print('  Tipo: Choropleth map → Europa')
print('  Columna región: cntry (ISO 2 letras)')
print('  Columna valor:  BRECHA_GNDR')
print('  Tooltip:        wb_hombres, wb_mujeres, n_total, country_name')
print('  Paleta:         divergente azul→rojo (negativo→positivo)')
print('  Highlight:      ES (color/borde diferenciado)')

RESUMEN V1 — MAPA DE LA PARADOJA

Países representados: 30
Rango brecha: [-0.047, 0.281]
Mediana brecha: 0.100

Top 5 mayor brecha:
cntry country_name  wb_hombres  wb_mujeres  BRECHA_GNDR
   PT     Portugal       6.312       6.031        0.281
   CY       Cyprus       6.490       6.213        0.277
   ES        Spain       6.562       6.357        0.205
   SK     Slovakia       6.232       6.052        0.180
   IT        Italy       6.337       6.163        0.174

Top 5 menor brecha (o invertida):
cntry country_name  wb_hombres  wb_mujeres  BRECHA_GNDR
   IE      Ireland       6.654       6.701       -0.047
   LV       Latvia       5.865       5.897       -0.032
   IS      Iceland       6.633       6.639       -0.007
   FI      Finland       6.534       6.532        0.002
   ME   Montenegro       6.083       6.073        0.010

Configuración Flourish:
  Tipo: Choropleth map → Europa
  Columna región: cntry (ISO 2 letras)
  Columna valor:  BRECHA_GNDR
  Tooltip:        wb_hombres, wb_mu